## Bigram vs Neural Network

In [18]:
import torch
import torch.nn.functional as F


In [19]:
words=open("/Users/mac/Desktop/Machine Learning/DL/models/data/names.txt", "r").read().splitlines()

print(words[:10])
print("Number of names:", len(words))

chars=sorted(list(set("".join(words))))

stoi={s:i for i, s in enumerate(chars)}
stoi["."]=0

itos={i:s for s, i in stoi.items()}
vocab_size=len(stoi)

print(vocab_size)

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']
Number of names: 32033
27


## Bigram model

In [20]:
import torch

N = torch.zeros((27,27), dtype=torch.int32)

for w in words:

    chs = ['.'] + list(w) + ['.']

    for ch1, ch2 in zip(chs, chs[1:]):

        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        N[ix1, ix2] += 1

In [21]:
P = (N + 1).float()
P /= P.sum(dim=1, keepdim=True)

In [22]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):

    out = []

    ix = 0

    while True:

        p = P[ix]

        ix = torch.multinomial(
            p,
            num_samples=1,
            replacement=True,
            generator=g
        ).item()

        if ix == 0:
            break

        out.append(itos[ix])

    print(''.join(out))

dry
n
ninilvil
dril
yienn
k
ni
n
jon
leel


In [23]:
# Compute Statistical Bigram Loss
log_likelihood=0.0

n=0
for w in words:
    chs=["."]+list(w)+["."]
    ix1=stoi[ch1]
    ix2=stoi[ch2]
    
    prob=P[ix1, ix2]
    
    log_likelihood+=torch.log(prob)
    
    n+=1
    
nll=-log_likelihood
loss=nll/n

print("Bigram loss:", loss.item())

Bigram loss: 0.994038999080658


## Bigram Training set

In [24]:
xs = []
ys = []

for w in words:

    chs = ['.'] + list(w) + ['.']

    for ch1, ch2 in zip(chs, chs[1:]):

        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(xs.shape)
print(ys.shape)

torch.Size([228146])
torch.Size([228146])


In [25]:
# Initialize Weights
g = torch.Generator().manual_seed(2147483647)

W = torch.randn(
    (27, 27),
    generator=g,
    requires_grad=True
)

In [26]:
# Forward Pass Test
xenc=F.one_hot(xs, num_classes=27).float()
logits=xenc@W

counts=logits.exp()
probs=counts/counts.sum(1, keepdim=True)

print(probs.shape)

torch.Size([228146, 27])


In [27]:
# Initial Loss
loss=-probs[torch.arange(len(xs)), ys].log().mean()

print(loss.item())

3.790374994277954


In [28]:
# Training Loop
for k in range(100):
    # Forward pass
    xenc=F.one_hot(xs, num_classes=27).float()
    
    logits=xenc@ W
    counts=logits.exp()
    probs=counts/counts.sum(1, keepdim=True)
    
    loss=-probs[torch.arange(len(xs)), ys].log().mean()
    
    # Backward pass
    W.grad=None
    loss.backward()
    
    # update
    W.data+=-50*W.grad
    
    if k%10==0:
        print(f"Eoch {k}: Loss={loss.item():.4f}")
        

Eoch 0: Loss=3.7904
Eoch 10: Loss=2.5543
Eoch 20: Loss=2.4665
Eoch 30: Loss=2.4305
Eoch 40: Loss=2.4118
Eoch 50: Loss=2.4004
Eoch 60: Loss=2.3928
Eoch 70: Loss=2.3872
Eoch 80: Loss=2.3830
Eoch 90: Loss=2.3798


In [29]:
# Final loss
print("Final Loss:", loss.item())

Final Loss: 2.368978261947632


In [30]:
## Generate names
g = torch.Generator().manual_seed(2147483647)

for _ in range(20):

    out = []

    ix = 0

    while True:

        xenc = F.one_hot(
            torch.tensor([ix]),
            num_classes=27
        ).float()

        logits = xenc @ W

        probs = F.softmax(logits, dim=1)

        ix = torch.multinomial(
            probs,
            num_samples=1,
            replacement=True,
            generator=g
        ).item()

        if ix == 0:
            break

        out.append(itos[ix])

    print(''.join(out))

dry
n
ninilvil
n
kl
yienn
r
ni
n
jon
leel
li
e

sh
y
kvikh
is
h
ury
